In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

# --------------------------------------
# 1. LOAD DATA
# --------------------------------------

# Load resume dataset
resumes_df = pd.read_csv("/content/ml_resume_dataset_4500.csv")

print(resumes_df.head(20))

# Load job description text
# with open("job_description.txt", "r", encoding="utf-8") as file:
#     job_description = file.read()
# Placeholder for job description. Please replace with your actual job description or upload 'job_description.txt'
job_description = """
We are looking for a highly skilled Machine Learning Engineer to join our innovative team.
 The ideal candidate will have strong experience in Python, machine learning algorithms,
 data preprocessing, model deployment, and cloud platforms like AWS or Google Cloud.
 Experience with deep learning frameworks such as TensorFlow or PyTorch is a plus.
 Excellent problem-solving skills and a passion for building scalable ML solutions are essential.
"""

# --------------------------------------
# 2. DATA PREVIEW
# --------------------------------------

print(resumes_df.head())
print("Total Candidates:", len(resumes_df))

# --------------------------------------
# 3. TEXT PREPROCESSING (BASIC)
# --------------------------------------

# Combine job description with resumes
documents = [job_description] + resumes_df["raw_text"].tolist()

# Convert text into numerical vectors using TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=500
)

tfidf_matrix = vectorizer.fit_transform(documents)

# --------------------------------------
# 4. SIMILARITY COMPUTATION
# --------------------------------------

# Calculate cosine similarity between job description and resumes
similarity_scores = cosine_similarity(
    tfidf_matrix[0:1],    # Job description vector
    tfidf_matrix[1:]      # Resume vectors
).flatten()

# --------------------------------------
# 5. ADD SCORES TO DATAFRAME
# --------------------------------------

resumes_df["Skill_Match_Score"] = similarity_scores

# Normalize experience
resumes_df["Experience_Score"] = resumes_df["years_experience"] / resumes_df["years_experience"].max()

# Final weighted score
resumes_df["Final_Score"] = (
    0.7 * resumes_df["Skill_Match_Score"] +
    0.3 * resumes_df["Experience_Score"]
)

# --------------------------------------
# 6. RANK CANDIDATES
# --------------------------------------

ranked_candidates = resumes_df.sort_values(
    by="Final_Score", ascending=False
)

print("\nTop 20 Recommended Candidates:")
print(ranked_candidates[["id", "Final_Score"]].head(20))

# --------------------------------------
# 7. VISUALIZATION
# --------------------------------------

plt.figure(figsize=(12, 6)) # Adjust figure size for better readability with more bars
plt.bar(
    ranked_candidates["id"][:20].astype(str), # Convert IDs to string for categorical x-axis
    ranked_candidates["Final_Score"][:20]
)
plt.title("Top 20 Candidate Ranking")
plt.xlabel("Candidate ID")
plt.ylabel("Final Score")
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()